# Clase 21 — PCA + K-Means + t-SNE: Reduccion de dimensiones y segmentacion

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador x UDLA

---

**Objetivos de hoy:**
1. Entender **por que** necesitamos PCA cuando tenemos muchas variables.
2. Aplicar PCA **paso a paso**: escalar, ajustar, interpretar componentes.
3. Combinar **PCA + K-Means** para segmentar paises por felicidad.
4. Interpretar clusters con **profiling** y visualizaciones.
5. Conocer **t-SNE** como alternativa no lineal para visualizacion.

**Dataset:** World Happiness Report — 153 paises, 12 columnas.

**SOLUCION**

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE

print("Imports OK")

---
## 1. Cargar y explorar los datos

In [ ]:
URL = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-21/world_happiness.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

In [ ]:
df.describe()

153 paises, 12 columnas. 7 indicadores numericos que usaremos para PCA.

In [ ]:
features = ["Economy", "Family", "Health", "Freedom", "Generosity", "Corruption", "Job Satisfaction"]
df_clean = df.dropna(subset=features).copy()
print(f"Paises despues de limpiar NaN: {len(df_clean)}")

In [ ]:
fig = px.scatter(
    df_clean, x="Economy", y="Health", color="Region",
    hover_data=["Country", "Happiness Score"],
    title="Economy vs Health por Region",
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Paises con mayor economia tienden a tener mejor salud. Pero esto es solo 2 de 7 variables. PCA nos permitira ver las 7 al mismo tiempo en un solo grafico.

---
## 2. La maldicion de la dimensionalidad

Con 7 variables necesitariamos **21 scatter plots** (combinaciones de 7 tomadas de 2 en 2). Cada uno muestra solo 2 de 7 dimensiones — una vista parcial.

**PCA (Principal Component Analysis)** comprime las 7 variables en 2-3 "super-variables" que capturan la **mayor variacion posible**. Es como tomar una foto 2D de un objeto 7D desde el mejor angulo.

**Necesitamos PCA.**

---
## 3. PCA paso a paso

### 3.1 Escalar (obligatorio)

In [ ]:
X = df_clean[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Antes de escalar:")
print(X.describe().loc[["mean", "std"]].round(2))
print("\nDespues de escalar:")
print(pd.DataFrame(X_scaled, columns=features).describe().loc[["mean", "std"]].round(2))

**StandardScaler es obligatorio para PCA** (misma razon que K-Means: trabaja con distancias y varianzas). Sin escalar, las variables con valores mas grandes dominarian los componentes.

### 3.2 Ajustar PCA (todos los componentes)

In [ ]:
pca = PCA()  # todos los componentes primero, para ver el scree plot
X_pca = pca.fit_transform(X_scaled)

print(f"Componentes: {pca.n_components_}")
print(f"Varianza explicada: {pca.explained_variance_ratio_.round(3)}")

### 3.3 Scree plot — Cuantos componentes necesitamos?

In [ ]:
var_exp = pca.explained_variance_ratio_
cum_var = np.cumsum(var_exp)

fig = px.bar(
    x=[f"PC{i+1}" for i in range(len(var_exp))], y=var_exp,
    title="Varianza explicada por componente",
    labels={"x": "Componente", "y": "Varianza explicada"},
    color_discrete_sequence=["#C82B40"])
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
fig = px.line(
    x=list(range(1, 8)), y=cum_var, markers=True,
    title="Varianza acumulada",
    labels={"x": "Numero de componentes", "y": "Varianza acumulada"})
fig.add_hline(y=0.7, line_dash="dash", annotation_text="70%", line_color="#16A34A")
fig.add_hline(y=0.8, line_dash="dash", annotation_text="80%", line_color="#EA580C")
fig.update_layout(template="plotly_white")
fig.show()

print(f"PC1: {var_exp[0]:.1%}")
print(f"PC1+PC2: {cum_var[1]:.1%}")
print(f"PC1+PC2+PC3: {cum_var[2]:.1%}")

**Interpretacion:** 2 componentes capturan ~71% de la varianza. Podemos representar 151 paises en un solo scatter 2D y conservar el 71% de la informacion de las 7 variables originales.

### 3.4 Que significan los componentes? — Loadings

In [ ]:
loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=features,
    columns=["PC1", "PC2", "PC3"])
print(loadings.round(3))

In [ ]:
fig = px.imshow(
    loadings.values,
    x=["PC1", "PC2", "PC3"],
    y=features,
    color_continuous_scale="RdBu_r", zmin=-0.6, zmax=0.6,
    title="Loadings: que significa cada componente",
    text_auto=".2f")
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion de loadings:**
- **PC1:** Economy, Health, Job Satisfaction y Family tienen loadings positivos altos -> PC1 representa **"desarrollo y calidad de vida"**.
- **PC2:** Generosity, Corruption y Freedom dominan -> PC2 representa **"libertad social y confianza"**.

El heatmap muestra de un vistazo que variables "pesan" en cada componente. Rojo = loading positivo alto, azul = negativo.

### 3.5 Visualizar paises en el espacio PCA

In [ ]:
df_clean["PC1"] = X_pca[:, 0]
df_clean["PC2"] = X_pca[:, 1]

fig = px.scatter(
    df_clean, x="PC1", y="PC2", color="Region",
    hover_data=["Country", "Happiness Score"],
    title="151 paises en 2D (PCA)",
    labels={"PC1": "PC1 — Desarrollo", "PC2": "PC2 — Libertad/confianza"},
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los paises se agrupan por region de forma natural. Europa Occidental (PC1 alto) vs Africa (PC1 bajo). Latinoamerica ocupa PC1 medio. Noruega aparece arriba-derecha (alto desarrollo). Chad abajo-izquierda. Ecuador en zona media, junto a otros latinoamericanos. PCA revelo esta estructura comprimiendo 7 variables en 2.

### 3.6 Encontrar paises especificos

In [ ]:
for country in ["Ecuador", "Norway", "Chad", "Costa Rica", "Japan"]:
    row = df_clean[df_clean["Country"] == country]
    if len(row) > 0:
        print(f"{country}: PC1={row['PC1'].values[0]:.2f}, PC2={row['PC2'].values[0]:.2f}, Region={row['Region'].values[0]}")

**Interpretacion:**
- **Noruega:** PC1 alto (muy desarrollado) y PC2 positivo (alta libertad/confianza). Arriba-derecha del grafico.
- **Chad:** PC1 muy bajo (bajo desarrollo). Abajo-izquierda.
- **Ecuador:** PC1 medio, cerca de otros paises latinoamericanos. Zona de transicion.
- **Costa Rica:** PC1 medio, PC2 relativamente alto (buena libertad para su nivel de desarrollo).
- **Japan:** PC1 alto (desarrollado) pero PC2 bajo (menor confianza social relativa).

### 3.7 Scatter 3D con 3 componentes

In [ ]:
pca3 = PCA(n_components=3)
X_pca3 = pca3.fit_transform(X_scaled)
df_clean["PC3"] = X_pca3[:, 2]

fig = px.scatter_3d(df_clean, x="PC1", y="PC2", z="PC3", color="Region",
    hover_data=["Country", "Happiness Score"], opacity=0.7,
    title=f"3 componentes ({pca3.explained_variance_ratio_.sum():.1%} varianza)")
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** 3 componentes capturan ~81% de la varianza. Rota el grafico para ver estructura que no era visible en 2D. PC3 agrega informacion adicional (generosity vs family/health).

---
## 4. PCA + K-Means: segmentacion en el espacio reducido

### 4.1 Silhouette para K=2..7

In [ ]:
sil_scores = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca[:, :2])
    sil_scores.append(silhouette_score(X_pca[:, :2], labels))

fig = px.line(
    x=list(range(2, 8)), y=sil_scores, markers=True,
    title="Silhouette Score por K (espacio PCA)",
    labels={"x": "K", "y": "Silhouette"})
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** K=3 tiene el mejor Silhouette Score. Tres segmentos es una division natural: paises desarrollados, en transicion y en desarrollo.

### 4.2 Ajustar K-Means con K=3

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean["cluster"] = km.fit_predict(X_pca[:, :2])

print(f"Paises por cluster:\n{df_clean['cluster'].value_counts().sort_index()}")

### 4.3 Visualizar clusters

In [ ]:
fig = px.scatter(
    df_clean, x="PC1", y="PC2",
    color=df_clean["cluster"].astype(str),
    hover_data=["Country", "Region", "Happiness Score"],
    title="3 segmentos de paises (PCA + K-Means)",
    labels={"PC1": "PC1 — Desarrollo", "PC2": "PC2 — Libertad"},
    color_discrete_sequence=["#2563EB", "#C82B40", "#16A34A"],
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los 3 clusters separan claramente: paises desarrollados (derecha), en transicion (centro) y en desarrollo (izquierda). K-Means encontro esta estructura sin que le dijeramos nada sobre regiones o niveles de desarrollo.

### 4.4 Coloreado por Happiness Score

In [ ]:
fig = px.scatter(
    df_clean, x="PC1", y="PC2",
    color="Happiness Score", color_continuous_scale="RdYlGn",
    hover_data=["Country", "Region"],
    title="Mismos datos, coloreados por Happiness Score")
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los clusters se alinean con la felicidad: el cluster de paises desarrollados (derecha) tiene los scores mas altos (verde). PCA + K-Means descubrio la estructura **sin nunca ver el Happiness Score**. Eso confirma que los 7 indicadores realmente explican la felicidad.

### 4.5 Usando sklearn Pipeline

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("kmeans", KMeans(n_clusters=3, random_state=42, n_init=10)),
])
df_clean["cluster_pipe"] = pipeline.fit_predict(X)

print(f"Clusters del pipeline:\n{df_clean['cluster_pipe'].value_counts().sort_index()}")

**Interpretacion:** Pipeline encadena todos los pasos (escalar -> PCA -> K-Means) en un solo objeto. Mismo resultado, codigo mas limpio y sin riesgo de olvidar un paso.

### 4.6 Perfil de clusters

In [ ]:
perfil = df_clean.groupby("cluster")[["Happiness Score"] + features].mean().round(2)
print(perfil)

for c in sorted(df_clean["cluster"].unique()):
    countries = df_clean[df_clean["cluster"] == c]["Country"].tolist()
    avg = df_clean[df_clean["cluster"] == c]["Happiness Score"].mean()
    print(f"\nCluster {c} ({len(countries)} paises, felicidad={avg:.1f}): {countries[:8]}...")

**Interpretacion del perfil:**
- **Cluster con Happiness alto:** Economy, Health, Family y Job Satisfaction altos -> **"Paises desarrollados"** (Noruega, Suiza, etc.).
- **Cluster medio:** Valores intermedios en todo -> **"Paises en transicion"** (Ecuador, Brasil, etc.).
- **Cluster bajo:** Economy y Health bajos -> **"Paises en desarrollo"** (Chad, Burundi, etc.).

PCA + K-Means descubrio grupos con niveles de felicidad muy distintos sin usar el Happiness Score como variable de entrada.

---
## 5. Ejercicio: Aplicar PCA + K-Means a Mall Customers

Usa el dataset de Mall Customers de la clase 19. Aplica PCA para reducir las 3 features (Age, income, spending) a 2 componentes. Luego aplica K-Means con K=5.

1. Carga los datos y escala las features.
2. Aplica PCA (todos los componentes). Cuanta varianza capturan 2 componentes?
3. Aplica PCA con 2 componentes + K-Means con K=5.
4. Visualiza el scatter en el espacio PCA coloreado por cluster.
5. Compara con los resultados de clase 19 (sin PCA). PCA agrega mucho con solo 3 variables?

In [ ]:
# SOLUCION

# Load Mall Customers
URL_MALL = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-19/mall_customers.csv"
df_mall = pd.read_csv(URL_MALL)
df_mall = df_mall.rename(columns={"Annual Income (k$)": "income", "Spending Score (1-100)": "spending"})

feat_mall = ["Age", "income", "spending"]
X_mall = df_mall[feat_mall]
X_mall_s = StandardScaler().fit_transform(X_mall)

pca_mall = PCA()
pca_mall.fit(X_mall_s)
print("Varianza por PC:", pca_mall.explained_variance_ratio_.round(3))
print(f"2 componentes: {sum(pca_mall.explained_variance_ratio_[:2]):.1%}")

X_mall_pca = PCA(n_components=2).fit_transform(X_mall_s)
km_mall = KMeans(n_clusters=5, random_state=42, n_init=10)
df_mall["cluster"] = km_mall.fit_predict(X_mall_pca)

fig = px.scatter(df_mall, x=X_mall_pca[:, 0], y=X_mall_pca[:, 1],
    color=df_mall["cluster"].astype(str), hover_data=["Age", "income", "spending"],
    title="Mall Customers en PCA",
    color_discrete_sequence=["#2563EB","#C82B40","#16A34A","#EA580C","#7C3AED"])
fig.update_layout(template="plotly_white")
fig.show()

print("\nCon solo 3 variables, PCA retiene ~78% en 2 componentes.")
print("Los clusters son similares a los de clase 19 (sin PCA).")
print("PCA agrega poco con pocas variables — brilla con 7+.")

**Interpretacion:** Con solo 3 variables originales, PCA retiene ~78% de la varianza en 2 componentes. Los clusters resultantes son similares a los obtenidos en clase 19 sin PCA. PCA agrega poco valor con pocas variables — su verdadero poder se nota con 7+ variables como en World Happiness.

---
## 6. t-SNE — Cuando PCA no alcanza

**PCA es lineal:** busca las direcciones de maxima varianza en linea recta. Funciona muy bien para datos con estructura lineal, pero puede fallar con datos complejos.

**t-SNE (t-distributed Stochastic Neighbor Embedding)** es **no lineal**: preserva las distancias entre vecinos cercanos, revelando clusters y estructuras que PCA no puede ver.

**Diferencia clave:** t-SNE no tiene `.transform()` — solo `.fit_transform()`. No es parametrico: no puedes aplicarlo a datos nuevos. Es solo para **visualizacion**.

### 6.2 t-SNE en World Happiness

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)
df_clean["tsne1"] = X_tsne[:, 0]
df_clean["tsne2"] = X_tsne[:, 1]

fig = px.scatter(df_clean, x="tsne1", y="tsne2", color="Region",
    hover_data=["Country", "Happiness Score"],
    title="t-SNE: World Happiness", opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** t-SNE separa las regiones de forma mas visual que PCA. Pero los ejes no tienen significado (no son "desarrollo" ni "libertad") y las distancias entre grupos lejanos no son confiables. Solo sirve para visualizar, no para preprocesar.

### 6.3 Introducción a imágenes como datos

Antes de ver t-SNE en acción, entendamos **cómo una imagen se convierte en datos numéricos**.

#### ¿Qué es una imagen para la computadora?

Una imagen **no es una foto**. Para la computadora es una **matriz de números**:

- **Escala de grises**: una matriz 2D. Cada celda es un pixel con un valor de intensidad (0 = negro, 255 = blanco).
- **Color (RGB)**: tres matrices 2D apiladas — una para Rojo, una para Verde, una para Azul. Cada pixel tiene 3 valores.

| Tipo | Dimensiones | Ejemplo 8×8 | Ejemplo 100×100 |
|------|-------------|-------------|------------------|
| Grises | alto × ancho | 64 valores | 10,000 valores |
| RGB | alto × ancho × 3 | 192 valores | 30,000 valores |

Para usar una imagen en ML, la **aplanamos**: la matriz se convierte en un **vector** (una fila del DataFrame). Una imagen de 8×8 = 64 columnas.

#### El dataset `digits` de sklearn

Es una versión simplificada del famoso **MNIST** (Modified National Institute of Standards and Technology, 1998). MNIST tiene 70,000 imágenes de 28×28 pixeles y se usa desde los años 90 para entrenar modelos de reconocimiento de dígitos. `digits` de sklearn es una versión reducida: 1,797 imágenes de 8×8 pixeles.

Cada imagen es un dígito (0-9) escrito a mano. Cada pixel tiene intensidad 0 (blanco) a 16 (negro).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print(f"Imágenes: {digits.data.shape[0]}")
print(f"Pixeles por imagen: {digits.data.shape[1]} (8x8)")
print(f"Clases: {sorted(set(digits.target))}")
print(f"\nPrimera imagen como vector (64 números):")
print(digits.data[0])

**Interpretación:** cada fila de `digits.data` son 64 números — los valores de intensidad de los 64 pixeles de una imagen de 8×8. Es exactamente igual a un DataFrame con 64 columnas numéricas.

In [ ]:
# Visualizar las imágenes — así se ven los dígitos
fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for digit in range(10):
    idx = np.where(digits.target == digit)[0][0]
    # Fila 1: imagen en grises
    axes[0, digit].imshow(digits.images[idx], cmap="gray_r", interpolation="nearest")
    axes[0, digit].set_title(str(digit), fontweight="bold", fontsize=13)
    axes[0, digit].axis("off")
    # Fila 2: heatmap de intensidades
    axes[1, digit].imshow(digits.images[idx], cmap="Reds", interpolation="nearest")
    axes[1, digit].axis("off")
plt.suptitle("Cada dígito es una imagen de 8x8 = 64 variables", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

**Interpretación:** cada dígito activa pixeles diferentes. Un "0" activa los bordes (forma circular), un "1" activa una columna vertical. La relación entre pixeles es **compleja y no lineal** — no hay una línea recta que separe un "3" de un "8" mirando solo 2 pixeles.

In [ ]:
# ¿Cómo se carga una imagen RGB en Python? (referencia)
# No lo ejecutamos aquí, pero así se hace:

# from PIL import Image  # pip install Pillow
# import numpy as np
#
# img = Image.open("foto.jpg")
# array = np.array(img)  # shape: (alto, ancho, 3) para RGB
# print(f"Dimensiones: {array.shape}")
# print(f"Una foto de 100x100 RGB = {100*100*3} valores")
#
# # Para ML: aplanar
# vector = array.flatten()  # shape: (30000,)

print("Referencia: así se carga una imagen RGB con PIL/Pillow")
print("Image.open('foto.jpg') → np.array → .flatten() → vector para ML")
print(f"\nDigits ya viene aplanado: {digits.data.shape}")

**Referencia para el futuro:** con `PIL` (Pillow) pueden cargar cualquier imagen (JPG, PNG) y convertirla en un array NumPy. Para escala de grises: `img.convert('L')`. Para RGB: 3 canales. El proceso siempre es: **cargar → convertir a array → aplanar → usar como fila de un DataFrame**.

Ahora que entendemos el dataset, veamos por qué PCA falla y t-SNE funciona con estas 64 dimensiones.

### 6.4 ¿Por qué las imágenes son NO lineales?

Si tomamos 2 pixeles cualesquiera (por ejemplo pixel 28 y pixel 43) y hacemos un scatter coloreado por dígito, los dígitos se **solapan completamente**. No hay ninguna línea recta que los separe mirando solo 2 pixeles.

La razón: la "identidad" de un dígito no está en un pixel individual, sino en la **combinación no lineal** de muchos pixeles. Un "0" no tiene un pixel más brillante que un "8" — tiene una **forma** distinta, y esa forma es una relación geométrica compleja entre pixeles.

In [ ]:
# Demostrar: 2 pixeles no alcanzan para separar dígitos
fig = px.scatter(x=digits.data[:, 28], y=digits.data[:, 43],
    color=digits.target.astype(str),
    title="Pixel 28 vs Pixel 43: los dígitos se solapan (relación NO lineal)",
    labels={"x": "Pixel 28 (centro)", "y": "Pixel 43 (abajo-derecha)"},
    color_discrete_sequence=["#2563EB","#C82B40","#16A34A","#EA580C","#7C3AED",
                             "#06B6D4","#D946EF","#F59E0B","#6B7280","#1E293B"],
    opacity=0.5)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** imposible separar dígitos con 2 pixeles. Todos se mezclan. PCA (que busca direcciones lineales) tampoco puede resolver esto bien en 2D. Necesitamos algo que capture la estructura **curva y compleja**: t-SNE.

### 6.5 PCA vs t-SNE en dígitos

In [ ]:
fig = px.scatter(x=X_dig_pca[:, 0], y=X_dig_pca[:, 1], color=digits.target.astype(str),
    title="PCA — digitos se solapan",
    color_discrete_sequence=["#2563EB","#C82B40","#16A34A","#EA580C","#7C3AED","#06B6D4","#D946EF","#F59E0B","#6B7280","#1E293B"],
    opacity=0.6)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
fig = px.scatter(x=X_dig_tsne[:, 0], y=X_dig_tsne[:, 1], color=digits.target.astype(str),
    title="t-SNE — 10 islas claras",
    color_discrete_sequence=["#2563EB","#C82B40","#16A34A","#EA580C","#7C3AED","#06B6D4","#D946EF","#F59E0B","#6B7280","#1E293B"],
    opacity=0.6)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** 64 dimensiones -> PCA pierde demasiado al comprimir a 2D, los digitos se solapan. t-SNE encuentra estructura no lineal y separa los 10 digitos perfectamente en 10 islas. **Usa t-SNE para VISUALIZAR, PCA para PREPROCESAR.**

### 6.4 Comparacion PCA vs t-SNE

| | PCA | t-SNE |
|---|---|---|
| Tipo | Lineal | No lineal |
| Preserva | Estructura global | Vecindarios locales |
| Parametrico | Si (.transform) | No (solo fit_transform) |
| Varianza explicada | Si | No |
| Velocidad | Rapido | Lento |
| Mejor para | Preprocesamiento + visualizacion | Visualizacion de datos complejos |

---
## Resumen de la clase

| Tema | Concepto clave |
|---|---|
| **El problema** | 7 variables = 21 scatter plots. Imposible visualizar todo |
| **PCA** | Comprime N variables en 2-3 "super-variables" que capturan la mayor varianza |
| **StandardScaler** | Obligatorio antes de PCA (trabaja con varianzas) |
| **Scree plot** | Muestra cuantos componentes necesitamos (buscar el codo) |
| **Loadings** | Dicen que significa cada componente (que variables pesan mas) |
| **PCA + K-Means** | PCA reduce dimensiones, K-Means segmenta en el espacio reducido |
| **Pipeline** | Encadena Scaler + PCA + K-Means en un solo objeto |
| **Profiling** | Calcular promedios por cluster para interpretar los segmentos |
| **t-SNE** | Reduccion no lineal — mejor para visualizar, no para preprocesar |
| **PCA vs t-SNE** | PCA = lineal, rapido, parametrico. t-SNE = no lineal, lento, solo visualizacion |